In [7]:
import pandas as pd

df = pd.read_csv("google_review_ratings.csv")

# Drop non-feature columns
df = df.drop(columns=["User", "Unnamed: 25"], errors="ignore")

# Convert to numeric
df = df.apply(pd.to_numeric, errors="coerce")

# Fill missing values
df = df.fillna(df.median())

# Create average rating per user (IMPORTANT)
df["avg_rating"] = df.mean(axis=1)

# Create target variable using median threshold
threshold = df["avg_rating"].median()
df["target"] = (df["avg_rating"] >= threshold).astype(int)

# Separate features and target
X = df.drop(columns=["avg_rating", "target"])
y = df["target"]

print("Threshold used:", threshold)
print(y.value_counts())


Threshold used: 2.0156250000000004
target
0    2728
1    2728
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(4092, 24) (1364, 24)


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled)
print(X_test_scaled)


[[-0.04373115 -0.60561056 -0.82398204 ... -0.22305004 -0.11192165
  -0.13456332]
 [ 0.70890355 -0.19324115 -0.35682707 ...  2.01716785  2.62212728
   0.06441796]
 [-0.79636584 -1.04593722 -0.76855687 ... -0.61508817 -0.58018418
  -0.67094765]
 ...
 [ 0.72085013 -0.18625184 -0.38058072 ...  0.20632505  0.43186708
   0.51428869]
 [-0.55743419  1.8755952  -0.36474495 ... -0.57152838 -0.45178961
  -0.48061773]
 [ 0.99562152 -0.02549767 -0.15888006 ...  0.38056422  0.64334048
   2.98857768]]
[[-1.74014585 -1.05991584 -0.79231052 ... -1.09424589 -1.1541834
  -1.33710238]
 [-0.10346406 -0.6265785  -0.84773568 ... -0.24171852 -0.13457952
  -0.1605174 ]
 [ 0.44607873 -0.17926253 -0.33307343 ...  2.01716785  2.62212728
   0.08172068]
 ...
 [-0.13930381 -0.67550368 -0.75272111 ... -0.78310451 -0.3762634
  -0.1605174 ]
 [-0.31850254  1.8755952  -0.04011184 ... -0.5528599  -0.42157913
  -0.44601229]
 [ 1.41375191  0.23310688 -0.03219396 ...  1.17086331  1.64783912
   0.95550805]]


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize Decision Tree (baseline)
dt = DecisionTreeClassifier(random_state=42)

# Train model
dt.fit(X_train, y_train)

# Predict on test data
y_pred_dt = dt.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_dt))

Accuracy: 0.8460410557184751

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.86      0.85       682
           1       0.86      0.83      0.84       682

    accuracy                           0.85      1364
   macro avg       0.85      0.85      0.85      1364
weighted avg       0.85      0.85      0.85      1364



In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

depths = [3, 5, 7, 10, None]

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"max_depth={d}, Accuracy={acc:.4f}")


max_depth=3, Accuracy=0.7192
max_depth=5, Accuracy=0.7749
max_depth=7, Accuracy=0.8394
max_depth=10, Accuracy=0.8526
max_depth=None, Accuracy=0.8460


In [12]:
splits = [2, 5, 10, 20]

for s in splits:
    dt = DecisionTreeClassifier(min_samples_split=s, random_state=42)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"min_samples_split={s}, Accuracy={acc:.4f}")


min_samples_split=2, Accuracy=0.8460
min_samples_split=5, Accuracy=0.8468
min_samples_split=10, Accuracy=0.8563
min_samples_split=20, Accuracy=0.8548


In [13]:
leaves = [1, 5, 10, 20]

for l in leaves:
    dt = DecisionTreeClassifier(min_samples_leaf=l, random_state=42)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"min_samples_leaf={l}, Accuracy={acc:.4f}")


min_samples_leaf=1, Accuracy=0.8460
min_samples_leaf=5, Accuracy=0.8497
min_samples_leaf=10, Accuracy=0.8607
min_samples_leaf=20, Accuracy=0.8504


In [15]:
dt_best = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=10,
    random_state=42
)

dt_best.fit(X_train, y_train)
y_pred = dt_best.predict(X_test)

print("Final Tuned Accuracy:", accuracy_score(y_test, y_pred))


Final Tuned Accuracy: 0.8548387096774194


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Baseline Random Forest
rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Baseline Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))


Baseline Random Forest Accuracy: 0.9002932551319648

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.91      0.90       682
           1       0.91      0.89      0.90       682

    accuracy                           0.90      1364
   macro avg       0.90      0.90      0.90      1364
weighted avg       0.90      0.90      0.90      1364



In [17]:
estimators = [50, 100, 200]

for n in estimators:
    rf = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"n_estimators={n}, Accuracy={acc:.4f}")



depths = [5, 10, 20, None]

for d in depths:
    rf = RandomForestClassifier(
        max_depth=d,
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"max_depth={d}, Accuracy={acc:.4f}")


leaves = [1, 5, 10]

for l in leaves:
    rf = RandomForestClassifier(
        min_samples_leaf=l,
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"min_samples_leaf={l}, Accuracy={acc:.4f}")


n_estimators=50, Accuracy=0.8944
n_estimators=100, Accuracy=0.9003
n_estimators=200, Accuracy=0.9062
max_depth=5, Accuracy=0.8270
max_depth=10, Accuracy=0.8922
max_depth=20, Accuracy=0.9018
max_depth=None, Accuracy=0.9003
min_samples_leaf=1, Accuracy=0.9003
min_samples_leaf=5, Accuracy=0.8922
min_samples_leaf=10, Accuracy=0.8666


In [18]:
rf_best = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=1,
    random_state=42
)

rf_best.fit(X_train, y_train)
y_pred_rf_best = rf_best.predict(X_test)

print("Final Tuned Random Forest Accuracy:",
      accuracy_score(y_test, y_pred_rf_best))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf_best))


Final Tuned Random Forest Accuracy: 0.906158357771261

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.91      0.91       682
           1       0.91      0.90      0.91       682

    accuracy                           0.91      1364
   macro avg       0.91      0.91      0.91      1364
weighted avg       0.91      0.91      0.91      1364



In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Baseline Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

print("Baseline Logistic Regression Accuracy:",
      accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))


Baseline Logistic Regression Accuracy: 0.9934017595307918

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       682
           1       0.99      1.00      0.99       682

    accuracy                           0.99      1364
   macro avg       0.99      0.99      0.99      1364
weighted avg       0.99      0.99      0.99      1364



In [20]:
C_values = [0.01, 0.1, 1, 10, 100]

for c in C_values:
    lr = LogisticRegression(C=c, max_iter=1000, random_state=42)
    lr.fit(X_train_scaled, y_train)
    y_pred = lr.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"C={c}, Accuracy={acc:.4f}")


C=0.01, Accuracy=0.9685
C=0.1, Accuracy=0.9853
C=1, Accuracy=0.9934
C=10, Accuracy=0.9963
C=100, Accuracy=0.9963


In [21]:
lr_best = LogisticRegression(
    C=10,
    max_iter=1000,
    random_state=42
)

lr_best.fit(X_train_scaled, y_train)
y_pred_lr_best = lr_best.predict(X_test_scaled)

print("Final Tuned Logistic Regression Accuracy:",
      accuracy_score(y_test, y_pred_lr_best))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr_best))


Final Tuned Logistic Regression Accuracy: 0.9963343108504399

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.99      1.00       682
           1       0.99      1.00      1.00       682

    accuracy                           1.00      1364
   macro avg       1.00      1.00      1.00      1364
weighted avg       1.00      1.00      1.00      1364



In [25]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings


warnings.filterwarnings("ignore")

# Baseline KNN
knn = KNeighborsClassifier()

knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

print("Baseline KNN Accuracy:",
      accuracy_score(y_test, y_pred_knn))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_knn))


Baseline KNN Accuracy: 0.8914956011730205

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.92      0.89       682
           1       0.92      0.86      0.89       682

    accuracy                           0.89      1364
   macro avg       0.89      0.89      0.89      1364
weighted avg       0.89      0.89      0.89      1364



In [23]:
k_values = [3, 5, 7, 9, 11, 15]

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"k={k}, Accuracy={acc:.4f}")


k=3, Accuracy=0.9047
k=5, Accuracy=0.8915
k=7, Accuracy=0.8981
k=9, Accuracy=0.8798
k=11, Accuracy=0.8754
k=15, Accuracy=0.8666


In [24]:
knn_best = KNeighborsClassifier(n_neighbors=3)

knn_best.fit(X_train_scaled, y_train)

y_pred_knn_best = knn_best.predict(X_test_scaled)

print("Final Tuned KNN Accuracy:",
      accuracy_score(y_test, y_pred_knn_best))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_knn_best))


Final Tuned KNN Accuracy: 0.9046920821114369

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.93      0.91       682
           1       0.93      0.88      0.90       682

    accuracy                           0.90      1364
   macro avg       0.91      0.90      0.90      1364
weighted avg       0.91      0.90      0.90      1364

